In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
batch_size = 128
latent_dim = 50 # Size of noise vector
epochs = 50
lr = 2e-4

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

train_dataset = datasets.MNIST(root=".", train=True, download=False, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [4]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024,1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024,784),
            nn.Tanh()
        )
    
    def forward(self, z):
        return self.net(z)

In [5]:
class Encoder(nn.Module):
    def __init__(self):
       super().__init__()
       self.net = nn.Sequential(
           nn.Linear(784, 1024),
           nn.ReLU(),
           nn.Linear(1024, 1024),
           nn.BatchNorm1d(1024),
           nn.ReLU(),
           nn.Linear(1024, latent_dim)
       ) 
    
    def forward(self, x):
        return self.net(x)

In [6]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784+latent_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024,1)
        )
    
    def forward(self, x, z):
        xz = torch.cat((x,z), dim = 1)
        return self.net(xz)

In [7]:
G = Generator().to(device)
E = Encoder().to(device)
D = Discriminator().to(device)

In [8]:
bce_loss = nn.BCEWithLogitsLoss()
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5,0.999))
optimizer_GE = optim.Adam(list(G.parameters()) + list(E.parameters()), lr=lr, betas=(0.5,0.999))

In [9]:
import os
k = 3
p = 1

os.makedirs("bigan_images", exist_ok=True)

In [10]:
for epoch in range(1, epochs+1):
    for i, (x_real, _) in enumerate(train_loader):
        x_real = x_real.to(device)
        batch_size = x_real.size(0)
        
        z_real = torch.rand(batch_size, latent_dim, device=device)
        
        for _ in range(p):
            G.eval()
            E.eval()
            D.train()
            
            x_fake = G(z_real).detach()
            z_fake = E(x_real).detach()
            
            D_real = D(x_real, z_fake)
            D_fake = D(x_fake, z_real)
            
            label_real = torch.ones_like(D_real)
            label_fake = torch.zeros_like(D_fake)
            
            loss_D = bce_loss(D_real, label_real) + bce_loss(D_fake, label_fake)
            
            optimizer_D.zero_grad()
            loss_D.backward()
            optimizer_D.step()
        
        for _ in range(k):
            G.train()
            E.train()
            D.eval()
            
            x_fake = G(z_real)
            z_fake = E(x_real)
            
            D_real = D(x_real, z_fake)
            D_fake = D(x_fake, z_real)
            
            loss_GE = bce_loss(D_real, label_fake) + bce_loss(D_fake, label_real)
            
            optimizer_GE.zero_grad()
            loss_GE.backward()
            optimizer_GE.step()
        
        if i % 100 == 0:
            print(f"Epoch [{epoch}/{epochs}] Batch {i}/{len(train_loader)}",
                  f"Loss D: {loss_D.item():.4f}, Loss GE: {loss_GE.item():.4f}")
    
    G.eval()
    with torch.no_grad():
        z = torch.randn(64, latent_dim, device=device)
        samples = G(z)
        samples = samples * 0.5 + 0.5
        save_image(samples, f"bigan_images/epoch_{epoch}.png", nrow=8)
    G.train()

Epoch [1/50] Batch 0/469 Loss D: 1.4168, Loss GE: 1.2711
Epoch [1/50] Batch 100/469 Loss D: 0.5453, Loss GE: 0.7417
Epoch [1/50] Batch 200/469 Loss D: 0.5145, Loss GE: 1.0517
Epoch [1/50] Batch 300/469 Loss D: 0.2541, Loss GE: 1.6744
Epoch [1/50] Batch 400/469 Loss D: 0.4203, Loss GE: 0.1772
Epoch [2/50] Batch 0/469 Loss D: 0.6207, Loss GE: 0.4957
Epoch [2/50] Batch 100/469 Loss D: 0.1363, Loss GE: 1.4787
Epoch [2/50] Batch 200/469 Loss D: 0.0610, Loss GE: 0.0564
Epoch [2/50] Batch 300/469 Loss D: 0.0575, Loss GE: 1.3578
Epoch [2/50] Batch 400/469 Loss D: 0.0988, Loss GE: 4.2478
Epoch [3/50] Batch 0/469 Loss D: 0.0659, Loss GE: 5.2359
Epoch [3/50] Batch 100/469 Loss D: 0.0335, Loss GE: 2.0936
Epoch [3/50] Batch 200/469 Loss D: 0.0387, Loss GE: 3.8374
Epoch [3/50] Batch 300/469 Loss D: 0.1166, Loss GE: 3.4792
Epoch [3/50] Batch 400/469 Loss D: 0.1810, Loss GE: 5.4338
Epoch [4/50] Batch 0/469 Loss D: 0.2367, Loss GE: 12.2872
Epoch [4/50] Batch 100/469 Loss D: 0.0827, Loss GE: 3.5387
Epoc

KeyboardInterrupt: 